# Semana 04
## Modelado de fuentes y validacion relacional

**Objetivo**: construir una fuente analítica tipo estrella, validar joins y exportar dimensiones/fact para Tableau.

**Herramientas teoricas de la semana**
- joins vs relationships
- cardinalidad
- tabla de hechos y dimensiones
- validacion de totales


### Agenda sugerida de 4 horas
- 0:00 - 0:30: cardinalidad y riesgos de duplicacion
- 0:30 - 1:20: construccion de modelo ligero con `pandas`
- 1:20 - 2:10: pruebas de integridad
- 2:10 - 3:20: export de fact y dimensions
- 3:20 - 4:00: discusion sobre uso en Tableau


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from _shared import (
    make_base_sales,
    introduce_quality_issues,
    profile_dataframe,
    clean_sales_data,
    build_star_schema,
    save_for_tableau,
    ensure_output_dir,
    contrast_ratio,
    make_high_dimensional_dataset,
)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
WEEK = "week-04"
OUTPUT_DIR = ensure_output_dir(WEEK)

clean, cleaning_log = clean_sales_data(introduce_quality_issues(make_base_sales(n=2200, seed=9), seed=9))
fact, dim_customer, dim_product, dim_geo, dim_date = build_star_schema(clean)
[fact.shape, dim_customer.shape, dim_product.shape, dim_geo.shape, dim_date.shape]


### Actividad 1. Validar la integridad del modelo
Antes de exportar, comprueba que el modelo no haya inflado filas o alterado ventas totales.


In [ ]:
validation = pd.DataFrame([
    {'check': 'sum_sales_original', 'value': clean['sales'].sum()},
    {'check': 'sum_sales_fact', 'value': fact['sales'].sum()},
    {'check': 'unique_customers_original', 'value': clean['customer_id'].nunique()},
    {'check': 'dim_customer_rows', 'value': len(dim_customer)},
])
validation


In [ ]:
joined = fact.merge(dim_customer, on=['customer_id', 'segment', 'channel'], how='left')
assert np.isclose(joined['sales'].sum(), fact['sales'].sum())
print('Join de control validado: no se inflan las ventas.')


### Actividad 2. Pensar el uso en Tableau
- `fact` para medidas transaccionales.
- `dim_date` para jerarquías temporales.
- `dim_geo` para mapas y comparaciones espaciales.
- `dim_product` para estructuras de categoría y subcategoría.


In [ ]:
for name, df in {
    'fact_orders': fact,
    'dim_customer': dim_customer,
    'dim_product': dim_product,
    'dim_geo': dim_geo,
    'dim_date': dim_date,
    'validation_checks': validation,
}.items():
    save_for_tableau(df, WEEK, name)


### Uso teorico de herramientas
- `merge()` traduce la teoría de joins y cardinalidad.
- `drop_duplicates()` ayuda a construir dimensiones estables.
- Las validaciones con sumas y conteos conectan modelado con veracidad analítica.
